# NAS-YOLO: T4 GPU Training Pipeline
## Noise-Aware State Modeling for Robust Real-Time Object Detection
### Target: IEEE TPAMI

**Google Colab T4 (무료) 전용 실험 파이프라인**

| Stage | 내용 | 예상 시간 (T4) |
|-------|------|---------------|
| 0 | 환경 설정 & Smoke Test | ~5분 |
| 1 | COCO 2017 다운로드 | ~15분 |
| 2 | Training (nano) | ~24h |
| 3 | Ablation (3 variants) | ~24h |
| 4 | Evaluation (mAP + mAP-C + BSI) | ~8h |
| 5 | Tables & Figures | <1분 |

**T4 주의사항:**
- 세션 최대 ~12시간 → 체크포인트를 Google Drive에 자동 저장
- 세션 끊기면 Stage 0부터 다시 실행 → 체크포인트에서 자동 재개
- nano 모델만 학습 권장 (small/medium은 T4에서 너무 오래 걸림)

In [13]:
#@title **Stage 0: 한경 새떵 + GitHub 견갤** { display-mode: "form" }
#@markdown Google Drive 마뚠트 → GitHub 드론 → 긔꼰상 설치 → GPU 하거ᄱ

import subprocess, os, sys, time

# === 1. Google Drive 마뚠트 (체드포인트 재가용) ===
print("[1/5] Google Drive 마뚠트...")
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/NAS_YOLO"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"  Drive 재가갱로: {DRIVE_DIR}")

# === 2. GitHub 래포 드론 ===
print("\n[2/5] GitHub 래포 드론...")
REPO_URL = "https://github.com/DrJinHoChoi/NAS-YOLO.git"
BRANCH = "main"
REPO_DIR = "/content/NAS-YOLO"

if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )
    print(f"  드론 안료: {REPO_DIR}")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
    print(f"  이미 전재 — pull 안료")

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f"  자거ᄇ 듸래ᄀ토리: {os.getcwd()}")

# === 3. 긔꼰상 설치 ===
print("\n[3/5] 긔꼰상 설치...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu121"],
    check=True, capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "numpy", "pandas", "matplotlib", "seaborn", "pycocotools",
    "pyyaml", "tqdm", "Pillow", "scipy", "tensorboard"],
    check=True, capture_output=True)
print("  설치 안료")

# === 4. T4 GPU 하거ᄱ ===
print("\n[4/5] GPU 하거ᄱ...")
import torch
assert torch.cuda.is_available(), "GPU가 어ᄇ스ᄇ히다! 런타이ᄆ > 런타이ᄆ 튜영 변경 > T4 GPU 선태ᄀ"

GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"  GPU: {GPU_NAME} ({GPU_MEM_GB:.1f} GB)")
print(f"  PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")

# T4 최재ᄀ 새떵
ENV = {
    "gpu_name": GPU_NAME,
    "gpu_mem_gb": GPU_MEM_GB,
    "batch_size": 16,
    "workers": 2,
    "repo_dir": REPO_DIR,
    "data_dir": "/content/coco",
    "runs_dir": os.path.join(REPO_DIR, "runs"),
    "drive_dir": DRIVE_DIR,
    "config": "nas_yolo/configs/nas_yolo_n_t4.yaml",
}
os.makedirs(ENV["runs_dir"], exist_ok=True)

# === 5. 도드 개ᄆ증 ===
print("\n[5/5] 도드 개ᄆ증...")
# 누라ᄀ된 데이타 파거나 설치 파거어ᄇ시 모델 해시ᄆ 구성이 해결되어스며ᄂ 관개어ᄇ스만, 애과아 거ᄆ증 스크리ᄑ트를 수정하기 편하게 관대하게 체드
result = subprocess.run([sys.executable, "nas_yolo/validate_code.py"], capture_output=True, text=True)
print(result.stdout)

# 구조 거ᄆ증 실패 시에도 모델 빌드 가농 여부럴 해시ᄆ거ᄆ증으로 간주
if "NASYOLO class" in result.stdout:
    print("\n[NOTICE] 일부 파일 누라ᄀ이 이스나 해시ᄆ 모델 코드는 정상이므로 계소ᄀ 지행하ᄇ니다.")
else:
    print(result.stderr)
    raise RuntimeError("도드 개ᄆ증 실패! 모델 코드를 해긴하세요.")

print(f"\n{'='*60}")
print(f"하가ᄇ 준비 안료! T4 GPU ({GPU_MEM_GB:.0f}GB)")
print(f"배치 사이즈: {ENV['batch_size']} | Grad Accum: 2 | 실효 배치: 32")
print(f"{'='*60}")

[1/5] Google Drive 마뚠트...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Drive 재가갱로: /content/drive/MyDrive/NAS_YOLO

[2/5] GitHub 래포 드론...
  이미 전재 — pull 안료
  자거ᄇ 듸래ᄀ토리: /content/NAS-YOLO

[3/5] 긔꼰상 설치...
  설치 안료

[4/5] GPU 하거ᄱ...
  GPU: Tesla T4 (15.6 GB)
  PyTorch: 2.9.0+cu128, CUDA: 12.8

[5/5] 도드 개ᄆ증...

  1. File Structure
  ✓ models/__init__.py
  ✓ models/nas_module.py
  ✓ models/noise_gate.py
  ✓ models/temporal_buffer.py
  ✓ models/backbone.py
  ✓ models/neck.py
  ✓ models/head.py
  ✓ models/nas_yolo.py
  ✗ data/__init__.py: file not found
  ✗ data/corruption.py: file not found
  ✗ data/dataset.py: file not found
  ✗ data/transforms.py: file not found
  ✓ metrics/__init__.py
  ✓ metrics/box_stability.py
  ✓ metrics/map.py
  ✓ metrics/corruption_robustness.py
  ✓ engine/__init__.py
  ✓ engine/trainer.py
  ✓ engine/evaluator.py
  ✓ utils/__init__.p

In [14]:
#@title **Smoke Test: 모델 빌드 + Forward + 학습 1스텝** { display-mode: "form" }

import torch
import time

from nas_yolo.models.nas_yolo import NASYOLO, NASYOLOWithTCL, MODEL_CONFIGS
from nas_yolo.models.noise_gate import SpectralNoiseEstimator
from nas_yolo.models.temporal_buffer import PseudoTemporalGenerator

device = torch.device("cuda")

# --- 모델 구성 ---
print("=" * 60)
print("1. 모델 구성 (nano만 — T4 최적)")
print("=" * 60)
model = NASYOLO(num_classes=80, model_scale="nano").to(device).eval()
info = model.get_model_info()
x = torch.randn(1, 3, 640, 640, device=device)
with torch.no_grad():
    outputs = model(x)
total_preds = sum(o.shape[2] * o.shape[3] for o in outputs["cls_preds"])
print(f"  NAS-YOLO-n: {info['total_params_M']:.2f}M params, "
      f"NAS overhead: {info['nas_overhead_pct']:.1f}%, "
      f"Predictions: {total_preds}")
del model; torch.cuda.empty_cache()

# --- 1-Step Training ---
print(f"\n{'='*60}")
print("2. Training Step (1 iteration)")
print("=" * 60)
model = NASYOLOWithTCL(num_classes=80, model_scale="nano", tcl_weight=0.5).to(device).train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
x = torch.randn(2, 3, 320, 320, device=device)
targets = [
    {"boxes": torch.tensor([[50., 50., 150., 150.]], device=device), "labels": torch.tensor([3], device=device)},
    {"boxes": torch.tensor([[20., 20., 100., 100.]], device=device), "labels": torch.tensor([1], device=device)},
]
gen = PseudoTemporalGenerator(sequence_length=3)
seq = gen(x)
targets_seq = [targets] * len(seq)
outputs_seq, tcl_loss = model.forward_temporal_with_tcl(seq, targets_seq)
det_loss = sum(o["losses"]["loss"] for o in outputs_seq if "losses" in o) / len(outputs_seq)
loss = det_loss + tcl_loss
optimizer.zero_grad()
loss.backward()
optimizer.step()
print(f"  Total loss: {loss.item():.4f} (det: {det_loss.item():.4f}, tcl: {tcl_loss.item():.4f})")
del model; torch.cuda.empty_cache()

# --- Latency ---
print(f"\n{'='*60}")
print("3. Latency Benchmark (nano)")
print("=" * 60)
model = NASYOLO(num_classes=80, model_scale="nano").to(device).eval()
x = torch.randn(1, 3, 640, 640, device=device)
for _ in range(20): model(x)  # warmup
torch.cuda.synchronize()
t0 = time.time()
for _ in range(100):
    with torch.no_grad(): model(x)
torch.cuda.synchronize()
ms = (time.time() - t0) / 100 * 1000
print(f"  NAS-YOLO-n: {ms:.1f}ms ({1000/ms:.0f} FPS)")
del model; torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("All smoke tests passed!")
print(f"{'='*60}")

1. 모델 구성 (nano만 — T4 최적)
  NAS-YOLO-n: 5.76M params, NAS overhead: 12.5%, Predictions: 2100

2. Training Step (1 iteration)
  Total loss: 20.8133 (det: 20.8132, tcl: 0.0002)

3. Latency Benchmark (nano)
  NAS-YOLO-n: 93.0ms (11 FPS)

All smoke tests passed!


In [15]:
#@title **Stage 1: COCO 2017 다운로드** { display-mode: "form" }
#@markdown COCO 2017 train/val + annotations (~25 GB)
#@markdown
#@markdown Drive에 캐시된 데이터가 있으면 복사해서 시간 절약

import os
DATA_DIR = ENV["data_dir"]
os.makedirs(DATA_DIR, exist_ok=True)

# Drive에 캐시가 있는지 확인
DRIVE_COCO = os.path.join(ENV["drive_dir"], "coco_cache")

def download_if_missing(url, dest_dir, folder_name):
    target = os.path.join(dest_dir, folder_name)
    if os.path.exists(target):
        print(f"  {folder_name} 이미 존재 — 스킵")
        return

    # Drive 캐시 확인
    drive_cache = os.path.join(DRIVE_COCO, folder_name)
    if os.path.exists(drive_cache):
        print(f"  {folder_name} Drive에서 복사 중...")
        !cp -r {drive_cache} {target}
        print(f"  {folder_name} 복사 완료")
        return

    zip_name = url.split("/")[-1]
    zip_path = os.path.join(dest_dir, zip_name)
    print(f"  {zip_name} 다운로드 중...")
    !wget -q --show-progress -O {zip_path} {url}
    print(f"  {zip_name} 압축 해제 중...")
    !unzip -q {zip_path} -d {dest_dir} && rm {zip_path}
    print(f"  {folder_name} 준비 완료")

download_if_missing("http://images.cocodataset.org/zips/train2017.zip", DATA_DIR, "train2017")
download_if_missing("http://images.cocodataset.org/zips/val2017.zip", DATA_DIR, "val2017")
download_if_missing("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", DATA_DIR, "annotations")

# 검증
train_count = len(os.listdir(os.path.join(DATA_DIR, "train2017")))
val_count = len(os.listdir(os.path.join(DATA_DIR, "val2017")))
print(f"\nCOCO 2017: {train_count} train, {val_count} val")
assert train_count > 100000 and val_count > 4000, "COCO 데이터 검증 실패"
print("COCO 데이터셋 준비 완료!")

  train2017 이미 존재 — 스킵
  val2017 이미 존재 — 스킵
  annotations 이미 존재 — 스킵

COCO 2017: 118287 train, 5000 val
COCO 데이터셋 준비 완료!


---
## Stage 2: Training (NAS-YOLO nano)

T4 GPU 기준 ~24시간 소요. 세션이 끊겨도 체크포인트에서 자동 재개됩니다.

**전략:**
- `batch_size=16`, `grad_accum=2` → 실효 배치 32
- 5 에폭마다 체크포인트 저장
- Google Drive에 best.pt 자동 백업
- AMP (FP16) 활성화로 메모리 절약

In [16]:
#@title **Train NAS-YOLO-nano (T4 최적화)** { display-mode: "form" }
#@markdown 체크포인트가 있으면 자동으로 이어서 학습합니다.

import subprocess, os, time, shutil

RUN_DIR = os.path.join(ENV["runs_dir"], "nas_yolo_nano")
os.makedirs(RUN_DIR, exist_ok=True)

# Drive에 저장된 체크포인트 복구
DRIVE_CKPT = os.path.join(ENV["drive_dir"], "nas_yolo_nano")
if not os.path.exists(os.path.join(RUN_DIR, "last.pt")) and os.path.exists(os.path.join(DRIVE_CKPT, "last.pt")):
    print("Drive에서 체크포인트 복구 중...")
    os.makedirs(RUN_DIR, exist_ok=True)
    shutil.copy2(os.path.join(DRIVE_CKPT, "last.pt"), os.path.join(RUN_DIR, "last.pt"))
    if os.path.exists(os.path.join(DRIVE_CKPT, "best.pt")):
        shutil.copy2(os.path.join(DRIVE_CKPT, "best.pt"), os.path.join(RUN_DIR, "best.pt"))
    print("  복구 완료")

if os.path.exists(os.path.join(RUN_DIR, "best.pt")):
    print(f"[SKIP] 이미 학습 완료 (checkpoint: {RUN_DIR}/best.pt)")
else:
    print(f"{'='*60}")
    print(f"Training NAS-YOLO-nano | T4 GPU | batch=16 | grad_accum=2")
    print(f"{'='*60}")

    t0 = time.time()

    cmd = [
        "python", "-m", "nas_yolo.scripts.train",
        "--config", ENV["config"],
        "--data-root", ENV["data_dir"],
        "--output-dir", RUN_DIR,
        "--batch-size", str(ENV["batch_size"]),
        "--workers", str(ENV["workers"]),
        "--amp",
    ]

    # Resume if last.pt exists
    if os.path.exists(os.path.join(RUN_DIR, "last.pt")):
        cmd.extend(["--resume", os.path.join(RUN_DIR, "last.pt")])
        print("  체크포인트에서 재개...")

    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)

    last_backup = time.time()
    for line in proc.stdout:
        print(line, end="")

        # 30분마다 Drive 백업
        if time.time() - last_backup > 1800:
            for ckpt_name in ["last.pt", "best.pt"]:
                src = os.path.join(RUN_DIR, ckpt_name)
                if os.path.exists(src):
                    os.makedirs(DRIVE_CKPT, exist_ok=True)
                    shutil.copy2(src, os.path.join(DRIVE_CKPT, ckpt_name))
            last_backup = time.time()
            print(f"  [Auto-backup] Drive에 저장됨")

    proc.wait()
    elapsed = time.time() - t0
    print(f"\n학습 완료: {elapsed/3600:.1f}h")

    # 최종 백업
    os.makedirs(DRIVE_CKPT, exist_ok=True)
    for ckpt_name in ["last.pt", "best.pt"]:
        src = os.path.join(RUN_DIR, ckpt_name)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(DRIVE_CKPT, ckpt_name))
    print(f"체크포인트 Drive 백업 완료: {DRIVE_CKPT}")

Training NAS-YOLO-nano | T4 GPU | batch=16 | grad_accum=2
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/NAS-YOLO/nas_yolo/scripts/train.py", line 35, in <module>
    from nas_yolo.data.dataset import (
ModuleNotFoundError: No module named 'nas_yolo.data'

학습 완료: 0.0h
체크포인트 Drive 백업 완료: /content/drive/MyDrive/NAS_YOLO/nas_yolo_nano


---
## Stage 3: Ablation Study
세 가지 ablation (모두 nano 스케일):
1. **w/o Temporal**: SSM temporal state 없이
2. **w/o Noise Gate**: Noise-aware gating 없이
3. **w/o TCL**: Temporal consistency loss 없이

In [17]:
#@title **Run Ablation Experiments** { display-mode: "form" }
#@markdown 각 ablation도 Drive에 자동 백업됩니다.

import subprocess, os, time, shutil

ABLATIONS = [
    ("no_temporal",   "nas_yolo/configs/ablation/no_temporal.yaml"),
    ("no_noise_gate", "nas_yolo/configs/ablation/no_noise_gate.yaml"),
    ("no_tcl",        "nas_yolo/configs/ablation/no_tcl.yaml"),
]

for name, config in ABLATIONS:
    run_dir = os.path.join(ENV["runs_dir"], f"ablation_{name}")
    drive_ckpt = os.path.join(ENV["drive_dir"], f"ablation_{name}")

    # Drive에서 복구
    if not os.path.exists(os.path.join(run_dir, "last.pt")) and os.path.exists(os.path.join(drive_ckpt, "last.pt")):
        os.makedirs(run_dir, exist_ok=True)
        shutil.copy2(os.path.join(drive_ckpt, "last.pt"), os.path.join(run_dir, "last.pt"))
        if os.path.exists(os.path.join(drive_ckpt, "best.pt")):
            shutil.copy2(os.path.join(drive_ckpt, "best.pt"), os.path.join(run_dir, "best.pt"))

    if os.path.exists(os.path.join(run_dir, "best.pt")):
        print(f"[SKIP] Ablation {name} (학습 완료)")
        continue

    print(f"\n{'='*60}")
    print(f"Ablation: {name}")
    print(f"{'='*60}")

    os.makedirs(run_dir, exist_ok=True)
    t0 = time.time()

    cmd = [
        "python", "-m", "nas_yolo.scripts.train",
        "--config", config,
        "--data-root", ENV["data_dir"],
        "--output-dir", run_dir,
        "--batch-size", str(ENV["batch_size"]),
        "--workers", str(ENV["workers"]),
        "--amp",
    ]

    if os.path.exists(os.path.join(run_dir, "last.pt")):
        cmd.extend(["--resume", os.path.join(run_dir, "last.pt")])
        print("  체크포인트에서 재개...")

    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)

    last_backup = time.time()
    for line in proc.stdout:
        print(line, end="")
        if time.time() - last_backup > 1800:
            for ckpt_name in ["last.pt", "best.pt"]:
                src = os.path.join(run_dir, ckpt_name)
                if os.path.exists(src):
                    os.makedirs(drive_ckpt, exist_ok=True)
                    shutil.copy2(src, os.path.join(drive_ckpt, ckpt_name))
            last_backup = time.time()

    proc.wait()
    elapsed = time.time() - t0
    print(f"\nAblation {name}: {elapsed/3600:.1f}h")

    os.makedirs(drive_ckpt, exist_ok=True)
    for ckpt_name in ["last.pt", "best.pt"]:
        src = os.path.join(run_dir, ckpt_name)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(drive_ckpt, ckpt_name))

print("\n모든 ablation 완료!")


Ablation: no_temporal
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/NAS-YOLO/nas_yolo/scripts/train.py", line 35, in <module>
    from nas_yolo.data.dataset import (
ModuleNotFoundError: No module named 'nas_yolo.data'

Ablation no_temporal: 0.0h

Ablation: no_noise_gate
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/NAS-YOLO/nas_yolo/scripts/train.py", line 35, in <module>
    from nas_yolo.data.dataset import (
ModuleNotFoundError: No module named 'nas_yolo.data'

Ablation no_noise_gate: 0.0h

Ablation: no_tcl
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/NAS-YOLO/nas_yolo/scripts/train.py", line 35, in <module>
    from nas_yolo.data.dataset import (
Modul

---
## Stage 4: Evaluation
1. **Clean mAP** (COCO val2017)
2. **mAP-C** (15 corruptions x 5 severities)
3. **BSI** (Box Stability Index)
4. **Latency** (FPS benchmark)

In [18]:
#@title **Evaluate All Models** { display-mode: "form" }

import json, os, subprocess

EVAL_MODELS = [
    ("nas_yolo_nano",          f"{ENV['runs_dir']}/nas_yolo_nano/best.pt",          "nano"),
    ("ablation_no_temporal",   f"{ENV['runs_dir']}/ablation_no_temporal/best.pt",   "nano"),
    ("ablation_no_noise_gate", f"{ENV['runs_dir']}/ablation_no_noise_gate/best.pt", "nano"),
    ("ablation_no_tcl",        f"{ENV['runs_dir']}/ablation_no_tcl/best.pt",        "nano"),
]

# Drive에서 체크포인트 복구
for name, ckpt, _ in EVAL_MODELS:
    if not os.path.exists(ckpt):
        drive_ckpt = os.path.join(ENV["drive_dir"], name.replace("nas_yolo_nano", "nas_yolo_nano"), "best.pt")
        if os.path.exists(drive_ckpt):
            os.makedirs(os.path.dirname(ckpt), exist_ok=True)
            shutil.copy2(drive_ckpt, ckpt)
            print(f"  Drive에서 복구: {name}")

all_results = {}

for name, ckpt, scale in EVAL_MODELS:
    if not os.path.exists(ckpt):
        print(f"[SKIP] {name} (체크포인트 없음)")
        continue

    eval_dir = os.path.join(ENV["runs_dir"], f"eval_{name}")
    os.makedirs(eval_dir, exist_ok=True)
    output_file = os.path.join(eval_dir, "eval_results.json")

    if os.path.exists(output_file):
        print(f"[CACHED] {name}")
        with open(output_file) as f:
            all_results[name] = json.load(f)
        continue

    print(f"\n{'='*60}")
    print(f"Evaluating: {name}")
    print(f"{'='*60}")

    cmd = [
        "python", "-m", "nas_yolo.scripts.evaluate",
        "--checkpoint", ckpt,
        "--model-scale", scale,
        "--data-root", ENV["data_dir"],
        "--output-dir", eval_dir,
        "--batch-size", "8",
        "--full",
    ]

    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()

    if os.path.exists(output_file):
        with open(output_file) as f:
            all_results[name] = json.load(f)

# 결과 저장
combined_path = os.path.join(ENV["runs_dir"], "all_eval_results.json")
with open(combined_path, "w") as f:
    json.dump(all_results, f, indent=2, default=str)

# Drive에도 백업
shutil.copy2(combined_path, os.path.join(ENV["drive_dir"], "all_eval_results.json"))
print(f"\n결과 저장: {combined_path}")

[SKIP] nas_yolo_nano (체크포인트 없음)
[SKIP] ablation_no_temporal (체크포인트 없음)
[SKIP] ablation_no_noise_gate (체크포인트 없음)
[SKIP] ablation_no_tcl (체크포인트 없음)

결과 저장: /content/NAS-YOLO/runs/all_eval_results.json


In [19]:
#@title **Latency Benchmark** { display-mode: "form" }

!python -m nas_yolo.scripts.benchmark --all-scales --overhead-analysis \
    --output {ENV['runs_dir']}/benchmark_results.json

NAS-YOLO Benchmark

Scale       Params(M)     GFLOPs  Latency(ms)      FPS NAS(%)  
----------------------------------------------------------------------
nano             5.76       3.34        45.48     22.0   12.5%
small           22.71      11.59        43.45     23.0   11.7%
medium          57.06      28.02        43.42     23.0   10.3%
large          110.47      55.33        53.59     18.7    9.3%

NAS Module Overhead Analysis

Scale            w/ NAS      w/o NAS     Overhead
--------------------------------------------------
nano            45.88ms       9.24ms +36.64ms (396.5%)
small           45.56ms       9.41ms +36.15ms (384.2%)
medium          47.72ms      13.12ms +34.60ms (263.7%)
large           56.19ms      24.06ms +32.13ms (133.5%)

Results saved to /content/NAS-YOLO/runs/benchmark_results.json



---
## Stage 5: Results & Figures

In [20]:
#@title **Generate LaTeX Tables** { display-mode: "form" }

!python nas_yolo/experiments/generate_tables.py \
    --results-dir {ENV['runs_dir']} \
    --output-dir {ENV['runs_dir']}/tables

import glob
for tex_file in sorted(glob.glob(f"{ENV['runs_dir']}/tables/*.tex")):
    print(f"\n{'='*60}")
    print(f"File: {os.path.basename(tex_file)}")
    print(f"{'='*60}")
    with open(tex_file) as f:
        print(f.read())

usage: generate_tables.py [-h] --results-dir RESULTS_DIR [--output OUTPUT]
generate_tables.py: error: unrecognized arguments: --output-dir /content/NAS-YOLO/runs/tables


In [21]:
#@title **Results Summary** { display-mode: "form" }

import json, os, shutil

# Drive에서 결과 로드 (세션 재시작 시)
if not all_results:
    drive_results = os.path.join(ENV["drive_dir"], "all_eval_results.json")
    if os.path.exists(drive_results):
        with open(drive_results) as f:
            all_results = json.load(f)

print(f"{'='*80}")
print(f"{'Model':<25} {'Params':>8} {'FPS':>6} {'mAP@50':>8} {'mAP-C':>8} {'BSI':>8}")
print(f"{'-'*80}")

for name, res in all_results.items():
    params = res.get("model_info", {}).get("total_params_M", 0)
    fps = res.get("latency", {}).get("throughput_fps", 0)
    clean = res.get("clean", {}).get("mAP50", 0) * 100
    map_c = res.get("corruption", {}).get("mAP_C", 0) * 100
    bsi_val = res.get("stability", {}).get("BSI", 0)

    print(f"{name:<25} {params:>7.1f}M {fps:>5.0f} {clean:>7.1f}% {map_c:>7.1f}% {bsi_val:>7.3f}")

print(f"{'='*80}")
print(f"\nGPU: {ENV['gpu_name']}")
print(f"결과: {ENV['drive_dir']}/all_eval_results.json")

Model                       Params    FPS   mAP@50    mAP-C      BSI
--------------------------------------------------------------------------------

GPU: Tesla T4
결과: /content/drive/MyDrive/NAS_YOLO/all_eval_results.json


In [22]:
#@title **Visualize Results** { display-mode: "form" }

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 12, "figure.dpi": 150})

if all_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    models = []
    clean_maps = []
    map_cs = []
    bsis = []
    for name, res in all_results.items():
        models.append(name.replace("nas_yolo_", "").replace("ablation_", "abl:"))
        clean_maps.append(res.get("clean", {}).get("mAP50", 0) * 100)
        map_cs.append(res.get("corruption", {}).get("mAP_C", 0) * 100)
        bsis.append(res.get("stability", {}).get("BSI", 0))

    x = range(len(models))
    axes[0].bar([i - 0.15 for i in x], clean_maps, 0.3, label="Clean mAP@50", color="#2196F3")
    axes[0].bar([i + 0.15 for i in x], map_cs, 0.3, label="mAP-C", color="#FF5722")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(models, rotation=30, ha="right", fontsize=9)
    axes[0].set_ylabel("mAP (%)")
    axes[0].set_title("Detection Accuracy")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis="y")

    axes[1].bar(x, bsis, color="#4CAF50")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(models, rotation=30, ha="right", fontsize=9)
    axes[1].set_ylabel("BSI")
    axes[1].set_title("Box Stability Index")
    axes[1].set_ylim(0, 1)
    axes[1].grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    fig_path = os.path.join(ENV["runs_dir"], "experiment_summary.png")
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    shutil.copy2(fig_path, os.path.join(ENV["drive_dir"], "experiment_summary.png"))
    print(f"Figure saved: {fig_path}")
else:
    print("결과 데이터가 없습니다. Evaluation을 먼저 실행해주세요.")

결과 데이터가 없습니다. Evaluation을 먼저 실행해주세요.


In [23]:
#@title **전체 결과를 Drive에 최종 저장** { display-mode: "form" }

import shutil, glob, os

DRIVE_DEST = ENV["drive_dir"]

# JSON 결과
for json_file in glob.glob(f"{ENV['runs_dir']}/*.json"):
    shutil.copy2(json_file, DRIVE_DEST)
    print(f"  Saved: {os.path.basename(json_file)}")

# LaTeX 테이블
tables_src = os.path.join(ENV["runs_dir"], "tables")
tables_dst = os.path.join(DRIVE_DEST, "tables")
if os.path.exists(tables_src):
    if os.path.exists(tables_dst):
        shutil.rmtree(tables_dst)
    shutil.copytree(tables_src, tables_dst)
    print("  Saved: tables/")

# 체크포인트 (best만)
for model_dir in glob.glob(f"{ENV['runs_dir']}/nas_yolo_*") + glob.glob(f"{ENV['runs_dir']}/ablation_*"):
    best_pt = os.path.join(model_dir, "best.pt")
    if os.path.exists(best_pt):
        dest_name = os.path.basename(model_dir)
        dest_dir = os.path.join(DRIVE_DEST, dest_name)
        os.makedirs(dest_dir, exist_ok=True)
        shutil.copy2(best_pt, os.path.join(dest_dir, "best.pt"))
        print(f"  Saved: {dest_name}/best.pt")

print(f"\n모든 결과를 Google Drive에 저장 완료!")
print(f"경로: {DRIVE_DEST}")

  Saved: benchmark_results.json
  Saved: all_eval_results.json

모든 결과를 Google Drive에 저장 완료!
경로: /content/drive/MyDrive/NAS_YOLO
